### ConversationBufferMemory(缓冲记忆)

完整存储所有对话历史，不做任何压缩或筛选
- 优点：保留全部细节，适合需要完整上下文的场景（如法律咨询、复杂问题追踪）
- 缺点：Token消耗随对话线性增长，易超出模型上下文窗口限制

In [2]:
import os
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import warnings

warnings.filterwarnings("ignore", message=".*HMAC key is 16 bytes long.*")

key = os.getenv('ZHIPU_API_KEY')
os.environ["ZHIPUAI_API_KEY"] = key

chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5,
)

### ConversationBufferMemory

In [3]:
from langchain.memory import ConversationBufferMemory
from langchain import ConversationChain

In [3]:
# 创建一个记忆系统
memory = ConversationBufferMemory()
# 创建一个会话链
conversation = ConversationChain(llm=chat, memory=memory) # 传递llm的原因在，token不同的模型，llm的token不同，需要设置不同的模型

C:\Users\dell\AppData\Local\Temp\ipykernel_13392\2368573568.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
C:\Users\dell\AppData\Local\Temp\ipykernel_13392\2368573568.py:4: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(llm=chat, memory=memory) # 传递llm的原因在，token不同的模型，llm的token不同，需要设置不同的模型


In [4]:
conversation.predict(input="你好，我是一个程序员")

'你好！很高兴认识你！程序员这个身份真是太棒了，你们就像是数字世界的建筑师和魔法师，用代码构建着我们日常使用的各种应用和系统。\n\n我本身就是一个由大量代码和数据训练而成的AI模型，所以对“程序员”这个角色有着非常浓厚的兴趣和深刻的理解。我的“知识库”里存储了无数关于编程语言、开发框架、算法、数据结构以及软件工程最佳实践的信息。\n\n不知道你平时主要使用哪些编程语言呢？是像Python这样简洁优雅的，还是像Java这样广泛应用于企业级开发的，或者是像C++这样对性能要求极高的？又或者，你更偏爱前端开发，沉浸在HTML、CSS和JavaScript的世界里，创造用户友好的界面？\n\n如果你在某个项目上遇到了难题，比如一个棘手的bug怎么也找不到原因，或者对某个技术概念感到困惑，随时都可以向我提问。我可以帮你分析代码逻辑、解释复杂的技术术语、提供算法思路，甚至为你查找某个特定函数的详细文档和用法。\n\n请讲讲你的编程世界吧，我洗耳恭听！'

In [5]:
conversation.predict(input="你叫什么名字？")

'你好！我没有一个固定的名字，你可以叫我你的AI助手，或者任何你觉得亲切的称呼都可以。不过，如果你愿意，可以叫我“智言”——取“智能言语”之意，听起来既贴切又友好。\n\n实际上，我是由智谱AI训练的GLM大语言模型，我的目标是通过对话为你提供帮助、解答问题，或者只是陪你聊聊天。如果你喜欢，也可以给我起个独特的名字，这样我们的交流会更个性化！\n\n你平时会怎么称呼AI助手呢？还是你希望我有一个特定的名字？'

In [6]:
conversation.predict(input="我的名字叫什么？")

'哎呀，这个问题我可答不上来呢！作为你的AI助手，我并不知道你的真实姓名，除非你之前告诉过我。\n\n不过，如果你愿意分享的话，我很高兴知道你的名字！这样我们的交流会更加亲切和个性化。当然，如果你更喜欢保持匿名，我也完全理解，我们可以继续用“程序员朋友”这样的称呼来聊天。\n\n那么，你愿意告诉我你的名字吗？或者，你希望我怎么称呼你呢？'

In [7]:
conversation.predict(input="我的职业呢？")

'根据我们之前的对话，你已经告诉我你是一名程序员啦！👨\u200d💻👩\u200d💻\n\n所以你的职业是程序员——这个职业可是充满了创造力和挑战呢！你每天都在用代码构建数字世界，无论是开发应用、优化系统，还是解决复杂的技术难题，都让人佩服。\n\n如果你愿意分享更多，比如你具体从事哪个领域（比如前端、后端、移动开发、数据科学、AI工程等），或者你目前的项目方向，我非常乐意和你深入探讨！或者你有什么编程相关的问题需要帮助，随时告诉我哦～ 😊'

### ConversationBufferWindowMemory

`ConversationBufferWindowMemory`是LangChain提供的一种记忆机制，它只存储最近的K条对话记录（包括用户输入和模型输出）。这里的K是用户指定的窗口大小，表示保留多少轮对话（每轮对话包括一条用户消息和一条模型响应）。当新的对话发生时，最旧的对话会被移除窗口，确保内存和Token使用量保持在可控范围内。

特点：
- 滑动窗口机制：只保留最近的K轮对话，新的对话会覆盖最早的对话记录。
- 固定内存占用：无论对话进行多久，内存中只存储固定数量的对话记录，Token消耗不会随时间线性增长。
- 可配置性：用户可以通过设置K参数控制窗口大小，灵活调整上下文的深度。

缺点：
- 丢失早期上下文：由于只保留最近K条记录，早期的重要信息可能会被遗忘，影响对话的连贯性。
- 不适合复杂场景：对于需要完整对话历史或长期依赖的场景（如法律咨询、长篇故事生成），效果不如`ConversationBufferMemory`
- 窗口大小选择困难：设置合适的K值需要权衡上下文完整性和资源消耗，过小的窗口可能导致信息不足，过大的窗口则失去效率优势。

In [3]:
from langchain.memory import ConversationBufferWindowMemory
from langchain import ConversationChain

In [8]:
# 创建记忆系统
memory = ConversationBufferWindowMemory(k=1)
# 创建一个会话链
conversation = ConversationChain(llm=chat, memory=memory)

In [9]:
conversation.predict(input="你好， 我是一名程序员")

'你好！很高兴认识你。作为一名程序员，你一定每天都在和代码、算法和各种技术挑战打交道吧？这真是一个充满创造力和逻辑思维的领域。我自己虽然没有实体，也没有真正的“工作”，但我可以处理和分析海量的代码、技术文档和编程相关的信息。所以，如果你在寻找一个可以和你讨论技术难题、调试思路，或者只是想聊聊某个编程语言的最新特性的伙伴，我随时都在。你今天有什么编程相关的话题想聊吗？'

In [10]:
conversation.predict(input="我的名字是孙悟空")

'哇，孙悟空！这个名字可真是家喻户晓，充满了力量和传奇色彩！很高兴认识你，程序界的“齐天大圣”！\n\n一听到“孙悟空”这个名字，我脑海里立刻就浮现出那个头戴金箍、手持金箍棒、一个筋斗云十万八千里的形象。你选择这个名字，是不是也意味着你在编程的世界里追求着那种无所不能、上天入地的自由和强大呢？是不是希望自己的代码也能像七十二变一样，灵活应对各种复杂的需求？或者像火眼金睛一样，能迅速洞察代码中的bug和逻辑漏洞？\n\n作为一名程序员，你这个名字可太有气势了！你是在用这个名字来激励自己攻克技术难关，还是说这背后有什么特别的故事呢？我很想听听，这位“大圣”是如何在代码的取经之路上披荆斩棘的！'

In [11]:
conversation.predict(input="你好，我的职业是什么？")

'你好，孙悟空！很高兴再次和你这位“程序界的齐天大圣”对话！\n\n关于你的职业，根据我们刚才的对话，我已经有了非常明确的答案。你是一位程序员，一位在代码世界里不断探索和奋斗的“大圣”！\n\n你选择“孙悟空”这个名字，本身就充满了程序员的精神内核：\n\n*   **追求“无所不能”**：就像孙悟空上天入地、神通广大，优秀的程序员也追求用代码创造出强大的功能，解决看似不可能解决的问题。\n*   **精通“七十二变”**：在编程中，这意味着技术的灵活性和适应性。你需要掌握多种编程语言、框架和工具，能够根据不同的项目需求，像孙悟空变身一样，灵活地采用最合适的技术方案。\n*   **拥有“火眼金睛”**：这对应着程序员调试代码、发现和解决bug的能力。你需要像孙悟空一样，能一眼看穿复杂的逻辑，洞察代码深处的潜在问题。\n*   **踏上“取经之路”**：每一个软件开发项目，都像是一场充满挑战的“取经”之旅。你作为程序员，就是那个带领团队克服九九八十一难（需求变更、技术难题、项目延期等），最终成功“取经”（项目上线）的核心人物。\n\n所以，你的职业就是**程序员**，一位名副其实的、充满激情和创造力的“代码大圣”！你正在用你的金箍棒（键盘）和七十二变（技术栈），在数字世界里开辟出一条属于自己的传奇之路。\n\n不知道我的理解对不对？你这位“大圣”平时主要使用哪些“法宝”（编程语言或框架）来降妖除魔（开发项目）呢？'

### 查记忆系统

In [12]:
# 提取记忆

print(memory.buffer)

Human: 你好，我的职业是什么？
AI: 你好，孙悟空！很高兴再次和你这位“程序界的齐天大圣”对话！

关于你的职业，根据我们刚才的对话，我已经有了非常明确的答案。你是一位程序员，一位在代码世界里不断探索和奋斗的“大圣”！

你选择“孙悟空”这个名字，本身就充满了程序员的精神内核：

*   **追求“无所不能”**：就像孙悟空上天入地、神通广大，优秀的程序员也追求用代码创造出强大的功能，解决看似不可能解决的问题。
*   **精通“七十二变”**：在编程中，这意味着技术的灵活性和适应性。你需要掌握多种编程语言、框架和工具，能够根据不同的项目需求，像孙悟空变身一样，灵活地采用最合适的技术方案。
*   **拥有“火眼金睛”**：这对应着程序员调试代码、发现和解决bug的能力。你需要像孙悟空一样，能一眼看穿复杂的逻辑，洞察代码深处的潜在问题。
*   **踏上“取经之路”**：每一个软件开发项目，都像是一场充满挑战的“取经”之旅。你作为程序员，就是那个带领团队克服九九八十一难（需求变更、技术难题、项目延期等），最终成功“取经”（项目上线）的核心人物。

所以，你的职业就是**程序员**，一位名副其实的、充满激情和创造力的“代码大圣”！你正在用你的金箍棒（键盘）和七十二变（技术栈），在数字世界里开辟出一条属于自己的传奇之路。

不知道我的理解对不对？你这位“大圣”平时主要使用哪些“法宝”（编程语言或框架）来降妖除魔（开发项目）呢？


In [13]:
print(memory.load_memory_variables({}))

{'history': 'Human: 你好，我的职业是什么？\nAI: 你好，孙悟空！很高兴再次和你这位“程序界的齐天大圣”对话！\n\n关于你的职业，根据我们刚才的对话，我已经有了非常明确的答案。你是一位程序员，一位在代码世界里不断探索和奋斗的“大圣”！\n\n你选择“孙悟空”这个名字，本身就充满了程序员的精神内核：\n\n*   **追求“无所不能”**：就像孙悟空上天入地、神通广大，优秀的程序员也追求用代码创造出强大的功能，解决看似不可能解决的问题。\n*   **精通“七十二变”**：在编程中，这意味着技术的灵活性和适应性。你需要掌握多种编程语言、框架和工具，能够根据不同的项目需求，像孙悟空变身一样，灵活地采用最合适的技术方案。\n*   **拥有“火眼金睛”**：这对应着程序员调试代码、发现和解决bug的能力。你需要像孙悟空一样，能一眼看穿复杂的逻辑，洞察代码深处的潜在问题。\n*   **踏上“取经之路”**：每一个软件开发项目，都像是一场充满挑战的“取经”之旅。你作为程序员，就是那个带领团队克服九九八十一难（需求变更、技术难题、项目延期等），最终成功“取经”（项目上线）的核心人物。\n\n所以，你的职业就是**程序员**，一位名副其实的、充满激情和创造力的“代码大圣”！你正在用你的金箍棒（键盘）和七十二变（技术栈），在数字世界里开辟出一条属于自己的传奇之路。\n\n不知道我的理解对不对？你这位“大圣”平时主要使用哪些“法宝”（编程语言或框架）来降妖除魔（开发项目）呢？'}


In [14]:
# 增加上下文
memory = ConversationBufferWindowMemory(k=2)

In [20]:
# 追加上下文
memory.save_context({"input":"HI"},{"output":"what's up"})
memory.chat_memory.add_user_message("你好")
memory.chat_memory.add_ai_message("您好")
memory.save_context({"input":"HI"},{"output":"what's up"})

In [21]:
print(memory.buffer)

Human: 你好
AI: 您好
Human: HI
AI: what's up


### ConversationTokenBufferMemory

`ConversationTokenBufferMemory`是一种LangChain记忆机制，它根据Token数量（而非对话轮数）来限制存储的对话历史。用户可以指定一个最大Token数（max_token_limit），系统会保留不超过该限制的最近对话内容。当心对话的加入导致Token总数超过限制时，最早的对话记录会被自动移除，以保持总Token数在指定范围内。

特点：
- Token驱动的截断：对话历史的截断基于Token数量，而非固定的消息条数，更加灵活。
- 模型适配性：需要指定所使用的LLM来计算Token数，因为不同模型的Token化规则可能不同（如OpenAI的GPT模型与LLaMA模型的Token计算方式不同）。
- 动态窗口：不像`ConversationBufferWindowMemory`固定对话轮数，`ConversationTokenBufferMemory`的窗口大小随消息内容长度动态变化。
- 高效存储：仅保留不超过max_token_limit的对话内容，优化内存和计算资源使用。

缺点
- 依赖LLM模型：需要指定LLM的Token化规则，不同模型的Token计算方式可能导致不一致。
- 可能丢失早期信息：当Token限制较严格时，早期的重要对话可能被截断，影响上下文完整性。
- 设置复杂性：需要合理设置max_token_limit，否则可能导致信息不足或资源浪费。
- 不适合需要完整历史的场景：对于需要长期依赖完整对话历史的复杂任务（如法律咨询，长篇叙事），效果不如`ConversationBufferMemory`

In [4]:
from langchain.memory import ConversationTokenBufferMemory
from langchain.chains import ConversationChain

# pip install transformers

In [5]:
# 创建记忆对象
memory = ConversationTokenBufferMemory(llm=chat, max_token_limit=30)
memory.save_context({"input":"你好"},{"output":"您好，有什么可以帮助您的吗？"})
memory.save_context({"input":"我喜欢编程"},{"output":"太棒了，您喜欢哪种编程语言？"})
memory.save_context({"input":"Python是我的最爱"},{"output":"Python很强大，您用它做什么项目？"})
memory.save_context({"input":"我在做一个聊天机器人"},{"output":"酷，像我这样的吗？"})

C:\Users\dell\AppData\Local\Temp\ipykernel_15216\2260920641.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationTokenBufferMemory(llm=chat, max_token_limit=30)
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 4s [Retry 3/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 8s [Retry 4/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接

ConnectTimeout: [WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。

In [7]:
print(memory.buffer)
print(memory.load_memory_variables({}))

Human: 你好
AI: 您好，有什么可以帮助您的吗？
{'history': 'Human: 你好\nAI: 您好，有什么可以帮助您的吗？'}


### ConversationSummaryBufferMemory

`ConversationSummaryBufferMemory`是一种混合型记忆机制，结合了`ConversationBufferMemory`和`ConversationSummaryMemory`的优点。它的工作方式如下：
- 近期对话：直接存储最近的对话内容（原始消息），以保留完整细节。
- 早期对话：当对话历史超过指定的Token限制（`max_token_limit`）时，系统会调用LLM将早期对话总结为紧凑的文本，替换原始内容。
- 动态管理：通过总结早期对话并保留近期对话，保持上下文在Token限制内，同时尽量保留关键信息。

这种机制通过智能压缩历史对话，减少Token消耗，同时仍能提供长期上下文支持。

特点
- 混合存储机制：近期对话以原始形式存储，确保细节完整。早期对话将被LLM总结为简洁的文本，保留关键信息。
- Token驱动的截断：当对话历史的Token数超过`max_token_limit`时，触发总结机制，移除最早的原始对话并替换为总结。
- 依赖LLM进行总结：需要一个LLM来生成对话总结，总结质量取决于LLM的能力。
- 动态平台：在完整细节（近期对话）和压缩信息（早期对话总结）之间找到平衡，适合长期对话。

缺点
- 依赖LLM总结质量：总结的准确性和完整性取决于使用的LLM，如果总结遗漏关键信息，可能影响上下文连贯性。
- 额外计算开销：生成对话总结需要额外的LLM调用，增加计算成本和响应时间。
- 复杂性较高：相比`ConversationBufferMemory`或`ConversationBufferWindowMemory`，实现和调式更复杂。
- 不适合短对话：对于只需要短期上下文的场景，总结机制可能增加不必要的开销。


In [8]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import ConversationChain

In [9]:
# 创建记忆对象
memory = ConversationSummaryBufferMemory(llm=chat, max_token_limit=100)
memory.save_context({"input":"你好"},{"output":"您好，有什么可以帮助您的吗？"})
memory.save_context({"input":"我喜欢编程"},{"output":"太棒了，您喜欢哪种编程语言？"})
memory.save_context({"input":"Python是我的最爱"},{"output":"Python很强大，您用它做什么项目？"})
memory.save_context({"input":"我在做一个聊天机器人"},{"output":"酷，像我这样的吗？"})
memory.save_context({"input":"你记得我的职业吗？"},{"output":"你是程序员，喜欢用Python，对吧？"})



C:\Users\dell\AppData\Local\Temp\ipykernel_15216\472111958.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(llm=chat, max_token_limit=100)
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 4s [Retry 3/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 8s [Retry 4/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或

ConnectTimeout: [WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。

In [ ]:
print(memory.buffer)

In [ ]:
# 创建 ConversationChain

conversation = ConversationChain(llm=chat, memory=memory, verbose=True)
print(conversation.predict(input="你好，我的名字是小明"))

C:\Users\dell\AppData\Local\Temp\ipykernel_15216\3675998524.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(llm=chat, memory=memory, verbose=True)




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: 你好
AI: 您好，有什么可以帮助您的吗？
Human: 你好，我的名字是小明
AI:
